# Lab Exercise 7 — K-Means vs K-Medoids Clustering
**Course:** MAI511-2 — Advanced Machine Learning  
**Dataset:** Wholesale Customers (440 records)  
**Aim:** Implement and compare K-Means and K-Medoids, determine a suitable number of clusters, and evaluate cluster quality.

## Report checklist
This notebook addresses every task and expected output in the supplied Lab Exercise 7 report:

- [x] Select a real-world dataset and identify numerical clustering features.
- [x] Inspect, clean and preprocess the data; scale the selected features.
- [x] Exclude target/class labels and non-feature identifiers from clustering.
- [x] Implement K-Means and determine K using the Elbow Method and Silhouette Score.
- [x] Implement K-Medoids and select K using clustering evaluation.
- [x] Identify and interpret K-Means centroids and K-Medoids medoids.
- [x] Visualize both clusterings.
- [x] Evaluate Silhouette Score, Davies–Bouldin Index and Calinski–Harabasz Index.
- [x] Compare the algorithms, centroid/medoid behavior and outlier influence.
- [x] Interpret cluster profiles and justify the final selection.
- [x] Include additional visualizations and self-learning/viva prompts.

> **Important:** The notebook calculates results from the supplied CSV when run. Do not copy expected numerical results from another run; use the values produced on your machine.

## 1. Dataset description and feature selection

The supplied CSV contains 440 wholesale-customer records and eight columns:
- `Channel` and `Region`: categorical/context columns, retained only for post-clustering interpretation.
- `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen`: six numerical annual spending variables used for clustering.

We deliberately exclude `Channel` and `Region` from model fitting so that the algorithms group customers by spending patterns rather than by pre-existing category codes.

In [ ]:
# If required, install dependencies in a notebook cell:
# %pip install pandas numpy matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    davies_bouldin_score, calinski_harabasz_score
)
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from mpl_toolkits.mplot3d import Axes3D  # enables 3D projection

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

In [ ]:
# Load the provided CSV.
# Keep the CSV in the same folder as this notebook, or change DATA_PATH.
DATA_PATH = "Wholesale customers data.csv"

try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    # Colab fallback: upload the CSV when prompted.
    try:
        from google.colab import files
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        df = pd.read_csv(uploaded_name)
    except Exception as exc:
        raise FileNotFoundError(
            "Place 'Wholesale customers data.csv' beside this notebook "
            "or update DATA_PATH to the correct file location."
        ) from exc

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Column information:")
display(df.info())

print("\\nMissing values:")
display(df.isna().sum().to_frame("Missing count"))

print("Duplicate rows:", df.duplicated().sum())
print("\\nDescriptive statistics:")
display(df.describe().T)

## 2. Data quality, preprocessing and exploratory visualizations

In [ ]:
# Separate clustering features from contextual categorical variables.
feature_cols = [
    "Fresh", "Milk", "Grocery", "Frozen",
    "Detergents_Paper", "Delicassen"
]
context_cols = [c for c in ["Channel", "Region"] if c in df.columns]

missing_features = [c for c in feature_cols if c not in df.columns]
if missing_features:
    raise ValueError(f"CSV is missing required features: {missing_features}")

X_raw = df[feature_cols].copy()

# Clean numeric values, impute missing values with the median if any,
# and remove exact duplicate rows if present.
X_raw = X_raw.apply(pd.to_numeric, errors="coerce")
imputer = SimpleImputer(strategy="median")
X_clean = pd.DataFrame(imputer.fit_transform(X_raw), columns=feature_cols, index=df.index)

duplicate_count = df.duplicated().sum()
if duplicate_count:
    keep_mask = ~df.duplicated()
    df = df.loc[keep_mask].reset_index(drop=True)
    X_clean = X_clean.loc[keep_mask].reset_index(drop=True)

print("Rows after duplicate handling:", len(X_clean))
print("Missing values after imputation:", int(X_clean.isna().sum().sum()))
print("Clustering features:", feature_cols)
print("Excluded context columns:", context_cols)

In [ ]:
# Visualization 1: distributions reveal skewness and scale differences.
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, feature_cols):
    sns.histplot(X_clean[col], kde=True, ax=ax, bins=30)
    ax.set_title(f"Distribution — {col}")
    ax.set_xlabel("Annual spending")
plt.suptitle("Spending Feature Distributions", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

# Visualization 2: boxplots expose extreme spending observations.
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, feature_cols):
    sns.boxplot(x=X_clean[col], ax=ax)
    ax.set_title(f"Outlier view — {col}")
plt.suptitle("Boxplots Before Scaling", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: correlation heatmap.
plt.figure(figsize=(10, 7))
corr = X_clean.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True)
plt.title("Correlation Between Wholesale Spending Features")
plt.tight_layout()
plt.show()

# Skewness table helps explain why outliers and scaling matter.
skew_table = X_clean.skew().sort_values(ascending=False).to_frame("Skewness")
display(skew_table)

### Scaling decision

Spending columns have different ranges and are typically right-skewed. StandardScaler is used for the primary experiment so each feature contributes on a comparable standardized scale. The original units are retained separately for interpreting centroids and medoids.

Outliers are not silently deleted in the main experiment: their influence is investigated explicitly later.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols, index=X_clean.index)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=X_clean, ax=axes[0])
axes[0].set_title("Before StandardScaler")
axes[0].tick_params(axis="x", rotation=35)
sns.boxplot(data=X_scaled_df, ax=axes[1])
axes[1].set_title("After StandardScaler")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

display(X_scaled_df.describe().T.round(3))

## 3. K-Means clustering

K-Means minimizes within-cluster sum of squared distances (inertia). We examine several K values using the Elbow Method and Silhouette Score. The elbow is a visual trade-off rather than an automatic proof of the best K; silhouette and interpretability are considered alongside it.

In [ ]:
k_values = range(2, 10)
kmeans_inertia = []
kmeans_silhouette = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = model.fit_predict(X_scaled)
    kmeans_inertia.append(model.inertia_)
    kmeans_silhouette.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(k_values), kmeans_inertia, marker="o")
axes[0].set_title("K-Means Elbow Method")
axes[0].set_xlabel("Number of clusters (K)")
axes[0].set_ylabel("Inertia (WCSS)")
axes[1].plot(list(k_values), kmeans_silhouette, marker="o")
axes[1].set_title("K-Means Silhouette Analysis")
axes[1].set_xlabel("Number of clusters (K)")
axes[1].set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "K": list(k_values),
    "Inertia": kmeans_inertia,
    "Silhouette": kmeans_silhouette
}).round(4))

## 4. K-Medoids implementation (PAM)

A medoid is an actual observation whose total dissimilarity to the other observations in its cluster is small. This notebook implements the Partitioning Around Medoids (PAM) swap procedure using Euclidean distances. It does not require the optional `scikit-learn-extra` package.

The implementation is suitable for this 440-row dataset. For very large datasets, use optimized or sampling-based K-Medoids implementations.

In [ ]:
from sklearn.metrics import pairwise_distances

def pam_kmedoids(X, n_clusters, max_iter=100, random_state=42):
    """Basic PAM K-Medoids using BUILD-like greedy initialization and SWAP refinement."""
    X = np.asarray(X, dtype=float)
    n = X.shape[0]
    if not 2 <= n_clusters < n:
        raise ValueError("n_clusters must be between 2 and n_samples - 1")

    D = pairwise_distances(X, metric="euclidean")
    rng = np.random.default_rng(random_state)

    # BUILD-style initialization: first medoid minimizes total distance.
    medoids = [int(np.argmin(D.sum(axis=1)))]
    while len(medoids) < n_clusters:
        current_cost = np.min(D[:, medoids], axis=1)
        best_candidate, best_cost = None, np.inf
        for candidate in range(n):
            if candidate in medoids:
                continue
            new_cost = np.minimum(current_cost, D[:, candidate]).sum()
            if new_cost < best_cost:
                best_cost, best_candidate = new_cost, candidate
        medoids.append(best_candidate)

    medoids = np.array(medoids, dtype=int)
    current_cost = np.min(D[:, medoids], axis=1).sum()

    # SWAP: try replacing each medoid with each non-medoid.
    for _ in range(max_iter):
        best_delta = 0.0
        best_swap = None
        nearest = D[:, medoids]
        current_nearest = nearest.min(axis=1)

        for medoid_pos in range(n_clusters):
            other_positions = [j for j in range(n_clusters) if j != medoid_pos]
            second_nearest = (
                nearest[:, other_positions].min(axis=1)
                if other_positions else np.full(n, np.inf)
            )

            # Evaluate every candidate replacement together (vectorized).
            proposed_costs = np.minimum(second_nearest[:, None], D).sum(axis=0)
            proposed_costs[medoids] = np.inf  # candidates already selected cannot be swapped in
            candidate = int(np.argmin(proposed_costs))
            delta = proposed_costs[candidate] - current_cost
            if delta < best_delta:
                best_delta = delta
                best_swap = (medoid_pos, candidate)

        if best_swap is None:
            break

        medoids[best_swap[0]] = best_swap[1]
        current_cost = np.min(D[:, medoids], axis=1).sum()

    # Stable labels based on nearest medoid.
    distances_to_medoids = D[:, medoids]
    labels = distances_to_medoids.argmin(axis=1)
    return labels, medoids, D

In [ ]:
# Evaluate K-Medoids across the same K range.
kmedoids_silhouette = []
kmedoids_cost = []
kmedoids_models = {}

for k in range(2, 9):
    labels, medoid_idx, D = pam_kmedoids(X_scaled, k, max_iter=100, random_state=RANDOM_STATE)
    kmedoids_models[k] = (labels, medoid_idx)
    kmedoids_silhouette.append(silhouette_score(X_scaled, labels))
    kmedoids_cost.append(np.min(D[:, medoid_idx], axis=1).sum())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(2, 9), kmedoids_cost, marker="o")
axes[0].set_title("K-Medoids Total Distance to Medoids")
axes[0].set_xlabel("Number of clusters (K)")
axes[0].set_ylabel("Total distance")
axes[1].plot(range(2, 9), kmedoids_silhouette, marker="o")
axes[1].set_title("K-Medoids Silhouette Analysis")
axes[1].set_xlabel("Number of clusters (K)")
axes[1].set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "K": list(range(2, 9)),
    "Total distance": kmedoids_cost,
    "Silhouette": kmedoids_silhouette
}).round(4))

### Selecting K

For a transparent, reproducible baseline, the notebook selects the K with the highest Silhouette Score for each algorithm from K=2…8. Review the elbow curve as well: if the silhouette optimum conflicts with a clear elbow or yields tiny, hard-to-interpret groups, justify a different K in your report rather than choosing mechanically.

In [ ]:
best_k_kmeans = int(list(k_values)[int(np.argmax(kmeans_silhouette))])
best_k_kmedoids = int(range(2, 9)[int(np.argmax(kmedoids_silhouette))])

print("Selected K for K-Means:", best_k_kmeans)
print("Selected K for K-Medoids:", best_k_kmedoids)

kmeans_model = KMeans(n_clusters=best_k_kmeans, random_state=RANDOM_STATE, n_init=30)
kmeans_labels = kmeans_model.fit_predict(X_scaled)
kmeans_centers_scaled = kmeans_model.cluster_centers_
kmeans_centers_original = scaler.inverse_transform(kmeans_centers_scaled)

kmedoids_labels, kmedoid_indices = kmedoids_models[best_k_kmedoids]
_, _, distance_matrix = pam_kmedoids(X_scaled, best_k_kmedoids, max_iter=100, random_state=RANDOM_STATE)
kmedoids_points_scaled = X_scaled[kmedoid_indices]
kmedoids_points_original = X_clean.iloc[kmedoid_indices].to_numpy()

print("K-Means cluster sizes:", pd.Series(kmeans_labels).value_counts().sort_index().to_dict())
print("K-Medoids cluster sizes:", pd.Series(kmedoids_labels).value_counts().sort_index().to_dict())

## 5. Dimensionality reduction and cluster visualizations

In [ ]:
# PCA is used only to project the six-dimensional feature space for visualization.
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca2 = pca2.fit_transform(X_scaled)
print("PCA variance explained by first two components:",
      round(pca2.explained_variance_ratio_.sum(), 4))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(x=X_pca2[:, 0], y=X_pca2[:, 1], hue=kmeans_labels,
                palette="tab10", s=45, ax=axes[0], legend="full")
centers_pca = pca2.transform(kmeans_centers_scaled)
axes[0].scatter(centers_pca[:, 0], centers_pca[:, 1], marker="X",
                s=250, c="black", edgecolor="white", label="Centroids")
axes[0].set_title(f"K-Means clusters (K={best_k_kmeans})")
axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left")

sns.scatterplot(x=X_pca2[:, 0], y=X_pca2[:, 1], hue=kmedoids_labels,
                palette="tab10", s=45, ax=axes[1], legend="full")
medoids_pca = pca2.transform(kmedoids_points_scaled)
axes[1].scatter(medoids_pca[:, 0], medoids_pca[:, 1], marker="D",
                s=180, c="black", edgecolor="white", label="Medoids")
axes[1].set_title(f"K-Medoids clusters (K={best_k_kmedoids})")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
for ax in axes:
    ax.set_xlabel("Principal Component 1")
    ax.set_ylabel("Principal Component 2")
plt.tight_layout()
plt.show()

In [ ]:
# 3D PCA visualization for both algorithms.
pca3 = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca3 = pca3.fit_transform(X_scaled)
fig = plt.figure(figsize=(16, 7))

ax1 = fig.add_subplot(121, projection="3d")
sc1 = ax1.scatter(X_pca3[:, 0], X_pca3[:, 1], X_pca3[:, 2],
                  c=kmeans_labels, cmap="tab10", s=25, alpha=0.75)
ax1.set_title("K-Means — 3D PCA")
ax1.set_xlabel("PC1"); ax1.set_ylabel("PC2"); ax1.set_zlabel("PC3")

ax2 = fig.add_subplot(122, projection="3d")
ax2.scatter(X_pca3[:, 0], X_pca3[:, 1], X_pca3[:, 2],
            c=kmedoids_labels, cmap="tab10", s=25, alpha=0.75)
ax2.scatter(*pca3.transform(kmedoids_points_scaled).T, marker="D",
            s=150, c="black", label="Medoids")
ax2.set_title("K-Medoids — 3D PCA")
ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2"); ax2.set_zlabel("PC3")
ax2.legend()
plt.tight_layout()
plt.show()

## 6. Cluster evaluation and comparison

In [ ]:
def evaluate_clustering(X, labels):
    return {
        "Silhouette Score (higher is better)": silhouette_score(X, labels),
        "Davies-Bouldin Index (lower is better)": davies_bouldin_score(X, labels),
        "Calinski-Harabasz Index (higher is better)": calinski_harabasz_score(X, labels)
    }

comparison = pd.DataFrame({
    "K-Means": evaluate_clustering(X_scaled, kmeans_labels),
    "K-Medoids": evaluate_clustering(X_scaled, kmedoids_labels)
}).T
comparison["K"] = [best_k_kmeans, best_k_kmedoids]
display(comparison.round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes, comparison.columns[:3]):
    sns.barplot(x=comparison.index, y=comparison[metric], ax=ax)
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=15)
plt.suptitle("K-Means vs K-Medoids — Evaluation Metrics", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette plots show the distribution of individual sample silhouette values.
def plot_silhouette(X, labels, title):
    sample_values = silhouette_samples(X, labels)
    fig, ax = plt.subplots(figsize=(9, 6))
    y_lower = 10
    for cluster_id in np.unique(labels):
        vals = np.sort(sample_values[labels == cluster_id])
        y_upper = y_lower + len(vals)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * len(vals), str(cluster_id))
        y_lower = y_upper + 10
    ax.axvline(sample_values.mean(), color="red", linestyle="--",
               label=f"Mean = {sample_values.mean():.3f}")
    ax.set_title(title)
    ax.set_xlabel("Silhouette coefficient")
    ax.set_ylabel("Cluster")
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_silhouette(X_scaled, kmeans_labels, "K-Means Silhouette Plot")
plot_silhouette(X_scaled, kmedoids_labels, "K-Medoids Silhouette Plot")

## 7. Centroids, medoids and cluster interpretation

- **Centroid:** arithmetic mean of all points in a K-Means cluster. It may not correspond to a real customer.
- **Medoid:** an actual customer record chosen to represent its cluster. It is less directly pulled by extreme observations than a mean, although K-Medoids is not immune to outliers.

The next cells show representatives in original spending units and cluster-level average profiles.

In [ ]:
kmeans_centroid_table = pd.DataFrame(kmeans_centers_original, columns=feature_cols)
kmeans_centroid_table.index = [f"Cluster {i}" for i in range(best_k_kmeans)]
kmedoid_table = X_clean.iloc[kmedoid_indices].copy()
kmedoid_table.index = [f"Medoid for cluster {i}" for i in range(best_k_kmedoids)]

print("K-Means centroids (original spending units):")
display(kmeans_centroid_table.round(1))
print("K-Medoids representative customer records (original spending units):")
display(kmedoid_table)

In [ ]:
# Cluster profiles: average spending by assigned cluster, in original units.
km_profile = X_clean.assign(Cluster=kmeans_labels).groupby("Cluster")[feature_cols].mean()
kmed_profile = X_clean.assign(Cluster=kmedoids_labels).groupby("Cluster")[feature_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.heatmap(np.log1p(km_profile), annot=True, fmt=".1f", cmap="YlGnBu",
            ax=axes[0], cbar_kws={"label": "log(1 + average spending)"})
axes[0].set_title("K-Means Cluster Profiles")
sns.heatmap(np.log1p(kmed_profile), annot=True, fmt=".1f", cmap="YlOrBr",
            ax=axes[1], cbar_kws={"label": "log(1 + average spending)"})
axes[1].set_title("K-Medoids Cluster Profiles")
plt.tight_layout()
plt.show()

# Cluster sizes.
size_df = pd.DataFrame({
    "K-Means": pd.Series(kmeans_labels).value_counts().sort_index(),
    "K-Medoids": pd.Series(kmedoids_labels).value_counts().sort_index()
}).fillna(0).astype(int)
display(size_df)
size_df.plot(kind="bar", figsize=(10, 5), title="Cluster Size Comparison")
plt.xlabel("Cluster label"); plt.ylabel("Number of customers")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# Optional context-only interpretation: compare Channel/Region composition after clustering.
# These columns were NOT used to train either clustering algorithm.
for algo_name, labels in [("K-Means", kmeans_labels), ("K-Medoids", kmedoids_labels)]:
    for context_col in context_cols:
        print(f"\\n{algo_name}: {context_col} composition (row-normalized)")
        display(pd.crosstab(labels, df[context_col], normalize="index").round(3))

## 8. Outlier influence analysis

In [ ]:
# Flag unusual observations with Isolation Forest on the scaled feature space.
# This is a diagnostic flag, not a ground-truth label.
iso = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
outlier_flag = iso.fit_predict(X_scaled)  # -1 = flagged outlier
print("Flagged observations:", int((outlier_flag == -1).sum()))
print("Flagged proportion:", round((outlier_flag == -1).mean(), 3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_pca2[:, 0], X_pca2[:, 1],
                c=np.where(outlier_flag == -1, "red", "steelblue"), alpha=0.7)
axes[0].set_title("Isolation Forest Outlier Flags on PCA Projection")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
sns.boxplot(data=X_clean, ax=axes[1])
axes[1].set_title("Feature Outlier Context")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
# Compare full-data clustering with a diagnostic run excluding flagged outliers.
# The comparison is performed at the same K for each algorithm.
inlier_mask = outlier_flag == 1
X_inliers = X_scaled[inlier_mask]

outlier_comparison = []
for name, k in [("K-Means", best_k_kmeans), ("K-Medoids", best_k_kmedoids)]:
    if name == "K-Means":
        labels_full = kmeans_labels
        labels_in = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=30).fit_predict(X_inliers)
    else:
        labels_full = kmedoids_labels
        labels_in, _, _ = pam_kmedoids(X_inliers, k, max_iter=100, random_state=RANDOM_STATE)

    outlier_comparison.append({
        "Algorithm": name,
        "K": k,
        "Full data silhouette": silhouette_score(X_scaled, labels_full),
        "Inliers-only silhouette": silhouette_score(X_inliers, labels_in)
    })

outlier_comparison_df = pd.DataFrame(outlier_comparison).set_index("Algorithm")
display(outlier_comparison_df.round(4))
print("Interpretation caution: the inliers-only scores are calculated on a different sample, "
      "so compare them as a sensitivity check, not as a perfectly like-for-like score.")

## 9. Automated result summary and conclusion guide

The following cell produces a data-driven summary. Use it as a starting point, then write your own interpretation of the cluster profiles, business meaning, and trade-offs. Do not claim one algorithm is universally superior.

In [ ]:
print("========== LAB 7 RESULT SUMMARY ==========")
print(f"Dataset: {len(X_clean)} customers, {len(feature_cols)} clustering features")
print(f"K-Means selected K: {best_k_kmeans}")
print(f"K-Medoids selected K: {best_k_kmedoids}")
print("\\nEvaluation comparison:")
display(comparison.round(4))

sil_km = comparison.loc["K-Means", "Silhouette Score (higher is better)"]
sil_md = comparison.loc["K-Medoids", "Silhouette Score (higher is better)"]
db_km = comparison.loc["K-Means", "Davies-Bouldin Index (lower is better)"]
db_md = comparison.loc["K-Medoids", "Davies-Bouldin Index (lower is better)"]
ch_km = comparison.loc["K-Means", "Calinski-Harabasz Index (higher is better)"]
ch_md = comparison.loc["K-Medoids", "Calinski-Harabasz Index (higher is better)"]

print("\\nMetric interpretation:")
print("Silhouette:", "K-Means higher" if sil_km > sil_md else "K-Medoids higher")
print("Davies-Bouldin:", "K-Means lower" if db_km < db_md else "K-Medoids lower")
print("Calinski-Harabasz:", "K-Means higher" if ch_km > ch_md else "K-Medoids higher")
print("\\nConclusion template:")
print("Choose the algorithm supported by the overall metric pattern, cluster sizes, "
      "visual separation, interpretability, and outlier sensitivity.")

## 10. Self-learning questions and viva preparation

1. Why must the spending features be scaled before distance-based clustering?
2. Why are `Channel` and `Region` excluded from model fitting but retained for interpretation?
3. What does inertia measure, and why does it never increase when K increases?
4. Why is the Elbow Method sometimes subjective?
5. What does a silhouette value near +1, 0, or -1 indicate?
6. Why are lower Davies–Bouldin values preferred?
7. Why are higher Calinski–Harabasz values preferred?
8. What is the key difference between a centroid and a medoid?
9. Why can K-Medoids be more robust to outliers than K-Means?
10. Does K-Medoids completely eliminate the effect of outliers? Explain.
11. Why is PCA used for visualization rather than as a replacement for the original features in this experiment?
12. What information is lost when six dimensions are projected into two dimensions?
13. Why might two clustering algorithms produce different cluster labels even when their partitions are similar?
14. How would you select K if silhouette and elbow results disagree?
15. What changes would be needed to scale PAM to tens of thousands of records?

### Final submission checklist
- [ ] Run all cells from top to bottom without errors.
- [ ] Confirm the CSV path and dataset dimensions.
- [ ] Save plots and tables generated by your actual run.
- [ ] Write a short interpretation for every final cluster.
- [ ] Justify K separately for K-Means and K-Medoids.
- [ ] Discuss centroid vs medoid and outlier influence.
- [ ] State the algorithm you prefer and support the decision with all three metrics.
- [ ] Save the notebook using the required naming convention from your report: `YourName_Last3Digits_Lab7.ipynb`.